# 3.0 Preprocesamiento del integrado (limpieza)

**KDD etapas 2–3 · CRISP-DM fase 3 · Sesión 3**

Esta unidad ejecuta el **pipeline de limpieza** sobre `data/03_processed/empleos.csv` — integrado construido por `src/limpieza/integrar_empleos.py` a partir de la **unión test + val** del dato crudo en CSV (train excluido) — siguiendo el orden canónico exigido en la sesión 3:

> **duplicados → categorías → nulos (banderas MCAR/MAR/MNAR) → outliers (IQR de Tukey) → validación final**

En lugar de reescribir el algoritmo aquí, se **reutilizan las funciones del módulo `src/limpieza/limpiar_empleos.py`** (criterio “funciones reutilizables + pruebas automáticas” de la rúbrica): cada fase termina con un `assert` que materializa los umbrales de aceptación (tabla 7.7 de la metodología).

In [1]:
from pathlib import Path
import sys

carpeta = Path.cwd()
while not (carpeta / "data").exists() and carpeta != carpeta.parent:
    carpeta = carpeta.parent
RAIZ = carpeta
PROC = RAIZ / "data" / "03_processed"
sys.path.insert(0, str(RAIZ / "src" / "limpieza"))

import pandas as pd
from limpiar_empleos import LimpiadorEmpleos, EXPECTED, LOGS_DIR
from dividir_por_outliers import main as dividir_por_outliers

print("Raíz del proyecto:", RAIZ)
print("pandas:", pd.__version__)

Raíz del proyecto: C:\Users\ACER\Desktop\MineriaProtect
pandas: 3.0.2


## §1 Instanciación (lectura defensiva del integrado)

Se lee el original **sin modificar nada**:

- `dtype=str` para conservar los ceros a la izquierda de los códigos ISCO/ESCO;
- `keep_default_na=False` + `na_values=[""]` para que solo la celda vacía sea `NaN` y los literales `'None'`, `'Present'`, `'unknown'` sigan siendo texto.

In [2]:
limpiador = LimpiadorEmpleos()
original = limpiador.original
print(f"Integrado cargado: {original.shape[0]:,} filas x {original.shape[1]} columnas "
      f"| {original['resume_id'].nunique():,} personas")

Integrado cargado: 376,567 filas x 11 columnas | 71,061 personas


## §2 Fase 1 · Diagnóstico (nada se toca)

In [3]:
limpiador.fase1_diagnostico()

FASE 1 DE 7 - DIAGNOSTICO (sobre el original, sin modificar)
Filas: 376,567 | Columnas: 11 | Personas: 71,061
Tipos de dato por columna:


,tipo
resume_id,str
start_date,str
end_date,str
university_level,str
matched_code,str
emparejado,str
occupation_code,str
occupation_label,str
isco_group,str
isco_group_label,str


Nulos por columna:
university_level    42807
occupation_code     28304
occupation_label    28304
isco_group          25805
isco_group_label    25805
isco_level          25805


Duplicados exactos: 0
Rango de fechas:


  start_date: 1965-2021


  end_date: 1970-2025


,resume_id,start_date,end_date,university_level,matched_code,emparejado,occupation_code,occupation_label,isco_group,isco_group_label,isco_level
0,10,Q1 2011,Q1 2015,Master,1222.1.2,ok,1222.1.2,public relations manager,1222,Advertising and public relations managers,4
1,10,Q1 2014,Q1 2015,Master,2310.1.10,ok,2310.1.10,communications lecturer,2310,University and higher education teachers,4
2,10,Q1 2015,Q3 2017,Master,2431.16,ok,2431.16,promotion assistant,2431,Advertising and marketing professionals,4
3,10,Q1 2016,Present,Master,2642.1.2,ok,2642.1.2,broadcast news editor,2642,Journalists,4
4,100013,Q1 2007,Q1 2008,NaN,9111.1,ok,9111.1,domestic cleaner,9111,Domestic cleaners and helpers,4
...,...,...,...,...,...,...,...,...,...,...,...
376562,99991,Q1 2014,Q1 2016,Secondary school,4227.1,ok,4227.1,market research interviewer,4227,Survey and market research interviewers,4
376563,99993,Q1 2009,Q1 2012,Master,3421.1,ok,3421.1,professional athlete,3421,Athletes and sports players,4
376564,99993,Q1 2009,Q1 2012,Master,unknown,unknown,NaN,NaN,NaN,NaN,NaN
376565,99993,Q1 2012,Q1 2013,Master,3422.4,ok,3422.4,sports instructor,3422,"Sports coaches, instructors and officials",4


## §3 Verificación independiente de las cifras del problema

Conteos directos con pandas sobre el original, para cruzar contra `EXPECTED` del módulo y contra el cuaderno `2.0`:

- duplicados **exactos**: 0;
- duplicados por clave corta `(resume_id, inicio, fin)`: 18.749 (pluriempleo);
- duplicados por clave corta `(resume_id, inicio, matched_code)`: 2.386.

In [4]:
exactos  = int(original.duplicated().sum())
corta    = int(original.duplicated(subset=["resume_id", "start_date", "end_date"]).sum())
corta_code = int(original.duplicated(subset=["resume_id", "start_date", "matched_code"]).sum())
print(f"Duplicados exactos (11 columnas)               : {exactos:,}")
print(f"Clave corta (resume_id, inicio, fin)            : {corta:,}  (pluriempleo simultáneo)")
print(f"Clave corta (resume_id, inicio, matched_code)   : {corta_code:,}")
print()
print("EXPECTED duplicado_pk (llave natural completa)  :", EXPECTED["duplicado_pk"])

Duplicados exactos (11 columnas)               : 0
Clave corta (resume_id, inicio, fin)            : 18,749  (pluriempleo simultáneo)
Clave corta (resume_id, inicio, matched_code)   : 2,386

EXPECTED duplicado_pk (llave natural completa)  : 0


## §4 Orden del pipeline y teoría de ausencias (sesión 3)

**¿Por qué duplicados primero?** Las duplicaciones se detectan sobre el *original* antes de cualquier transformación; si las banderas o re-categorizaciones vinieran antes, las filas idénticas podrían dejar de parecerlo y se perdería la detección.

**Mecanismos de ausencia en nuestros datos:**

| Ausencia | Mecanismo | Tratamiento (sesión 3) |
|---|---|---|
| Fechas futuras (>2026) | **MCAR** (error de captura, sin relación con el dato) | Eliminar (única eliminación) |
| Ocupación `'unknown'` / `NaN` | **MAR** (estructural: el mapeo no existe, no “falta el dato”) | Bandera, sin imputar |
| Educación `'None'` / celdas vacías | **MNAR** (el valor ausente está ligado al individuo) | Categoría propia `'No reportado'`, sin imputar moda |
| Empleo vigente `'Present'` | Censura por diseño | Bandera `es_vigente` |

**Decisión de no imputar:** imputar moda/mediana inventaría ocupaciones y educaciones para 25.805 y 42.807 filas; la minería de transiciones debe conservar la ausencia como *estado*, no disimularla.

### Fase 2 · Duplicados

In [5]:
limpiador.fase2_duplicados()

FASE 2 DE 7 - DUPLICADOS
Duplicados exactos: 0 | Por PK natural: 0
Claves cortas = pluriempleo/transicion real (filas extra si se deduplicara):
  (resume, start, end)   = 18,749 | (resume, start, code) = 2,386


### Fase 3 · Categorías y caso B (MNAR)

Re-verificación de cardinalidades canónicas (3 emparejados / 4 niveles educativos / 424 grupos ISCO / 2.855 etiquetas) y re-categorización de `'None'` / celdas vacías → `'No reportado'`.

In [6]:
limpiador.fase3_categorias()

FASE 3 DE 7 - CATEGORIAS (re-verificacion de cardinalidades)


,esperado,obtenido,ok
emparejado,3,3,True
university_level,4,4,True
isco_group_label,424,424,True
occupation_label,2855,2855,True



Distribucion emparejado:


,cantidad
emparejado,
ok,348263
unknown,25805
rescatado,2499


Distribucion university_level:


,cantidad
university_level,
Secondary school,147904
Bachelor,129507
Master,54697
NaN,42807
PhD,1652



Re-categorizacion caso B (MNAR): 'None' (0) + vacias (42,807) -> 'No reportado' (42,807)


,cantidad
university_level,
Secondary school,147904
Bachelor,129507
Master,54697
No reportado,42807
PhD,1652


### Fase 4 · Faltantes por bandera (MAR + censura)

Se crean `es_unknown_ocupacion`, `es_rescatado` y `es_vigente`. **Sin imputar y sin borrar**: cada `NaN` de ocupación queda explicado por una bandera.

In [7]:
limpiador.fase4_faltantes()

FASE 4 DE 7 - FALTANTES (banderas A-C, sin imputar)
es_unknown_ocupacion : 25,805 (personas: 16,055)
es_rescatado         : 2,499
es_vigente ('Present') : 18,956
occupation_code NaN  : 28,304 | explicados por bandera: 28,304
Personas con historial completo 'unknown': 605


### Fase 5 · Outliers como decisión de negocio (IQR de Tukey)

- Se calcula la duración trimestral inclusiva (`dur_Q = ordinal_fin - ordinal_ini + 1`; `'Present'` → `NaN`).
- Regla de Tukey sobre el distribuido de `dur_Q`: Q1=2, Q3=11, IQR=9 → límite superior **24,5**.
- Los 35.477 valores fuera de límite (24.687 personas) están **todos por la cola larga** (carreras de décadas). **Se conservan**: borrarlos segmentaría personas completas.
- Lo único que se elimina son las **0 fechas futuras** (no hay en test+val; si aparecieran, MCAR).

In [8]:
limpiador.fase5_outliers()

FASE 5 DE 7 - OUTLIERS
IQR (Tukey) sobre dur_Q: Q1=2.00 Q3=11.00 IQR=9.00 | limites [-11.50, 24.50]
Fuera de limites: 35,477 filas / 24,687 personas (todas por arriba)
|z| > 3 (solo comparacion): 8,460
Fechas futuras (anio > 2026): 0


Fechas futuras eliminadas: 0 | filas tras la fase: 376,567


## §5 Validación y exportación (fase 6 y 7)

La tabla de **umbrales de aceptación (6/6)** y la exportación del limpio. Comparación contra `EXPECTED`:

- `filas_limpio = 376.567` (0 fechas futuras en test+val);
- `filas_original` se conserva intacta (assert en exportación).

In [9]:
limpiador.fase6_validacion()
limpiador.fase7_exportar()

FASE 6 DE 7 - VALIDACION (tabla 7.7)


,criterio,valor,esperado,ok
0,Duplicados exactos,0,0,True
1,Duplicados por PK natural,0,0,True
2,Nulos restantes sin bandera,0,0,True
3,Nulos de ocupacion sin bandera,0,0,True
4,Fechas futuras,0,0,True
5,start > end,0,0,True


Impacto: 376,567 filas / 71,061 personas -> 376,567 filas / 71,061 personas (eliminadas: 0 filas por fechas futuras).
FASE 7 DE 7 - EXPORTAR


Escrito: C:\Users\ACER\Desktop\MineriaProtect\data\03_processed\empleos_limpio.csv (48.2 MB)
El CSV es la salida unica del bloque 4 (los codigos conservan sus ceros a la izquierda).
Columnas finales (14): resume_id, start_date, end_date, university_level, matched_code, emparejado, occupation_code, occupation_label, isco_group, isco_group_label, isco_level, es_unknown_ocupacion, es_rescatado, es_vigente
El archivo original data/03_processed/empleos.csv NO fue modificado.


## §6 Bitácora y trazabilidad

Cada decisión de limpieza queda en la bitácora de ejecución (qué, cuánto, método, razón, impacto) junto con una fila de trazabilidad con SHA-256 del original.

In [10]:
display(pd.DataFrame(limpiador.bitacora))
limpiador._persistir_bitacora_ejecucion()
print(f"Bitácora de ejecución: {LOGS_DIR / 'bitacora_limpieza_empleos.csv'}")
print(f"Estado final: {len(limpiador.df):,} filas | {limpiador.df['resume_id'].nunique():,} personas")

,columna,problema,cantidad,metodo,razon,impacto
0,university_level,'None' o celda vacia (MNAR),42807,Re-categorizar a 'No reportado',Imputar la moda (Secondary school) fabricaria ...,"Categoria propia, sin 'None' ni celdas vacias ..."
1,occupation_code/isco_*,NaN estructural (MAR),28304,Banderas es_unknown_ocupacion y es_rescatado,"El NaN dice 'sin emparejar', no 'sin dato'; im...","Conserva 16,055 personas (605 con historial co..."
2,end_date,'Present' (censura),18956,Bandera es_vigente,El empleo sigue y no tiene fin real; no se inv...,Duración indefinida marcada.
3,dur_Q (derivada),Cola larga (>=25 trimestres fuera del IQR),35477,Conservar (sin winsorizar),Carreras de ~40 anios son estabilidad real; bo...,0 filas eliminadas por atipicos.
4,start_date/end_date,Fechas con anio > 2026 (MCAR),0,Eliminar filas,Fecha posterior al anio del proyecto es imposi...,0 filas eliminadas.


Bitacora de ejecucion persistida: C:\Users\ACER\Desktop\MineriaProtect\logs\bitacora_limpieza_empleos.csv
Bitácora de ejecución: C:\Users\ACER\Desktop\MineriaProtect\logs\bitacora_limpieza_empleos.csv
Estado final: 376,567 filas | 71,061 personas


## §7 La cola larga como insumo de decisión (IQR, sesión 3)

La limpieza **conserva** los outliers de duración. Para cuantificar su impacto se genera una versión alternativa sin cola larga (`empleos_limpio_sin_outliers.csv`) con el script `dividir_por_outliers.py` (misma convención IQR: `dur_Q >= 25`).

In [11]:
dividir_por_outliers()

DIVISION POR OUTLIERS DE DURACION (empleos_limpio + sin_outliers)


Origen:  C:\Users\ACER\Desktop\MineriaProtect\data\03_processed\empleos_limpio.csv
Filas:   376,567 | Columnas: 14 | Personas: 71,061


IQR (Tukey) sobre dur_Q: Q1=2.00 Q3=11.00 IQR=9.00 | limites [-11.50, 24.50]
Outliers (dur_Q >= 25, cola larga): 35,477 filas / 24,687 personas
RESULTADO DE LA DIVISION
  Con outliers    : 376,567 filas / 71,061 personas  -> empleos_limpio.csv
  Sin outliers    : 341,090 filas / 69,182 personas  -> empleos_limpio_sin_outliers.csv
  Consistencia    : 376,567 - 35,477 outliers = 341,090 (sin outliers)


Escrito: C:\Users\ACER\Desktop\MineriaProtect\data\03_processed\empleos_limpio_sin_outliers.csv (43.6 MB)
El original data/03_processed/empleos_limpio.csv NO fue modificado.
Bitacora de ejecucion persistida: C:\Users\ACER\Desktop\MineriaProtect\logs\bitacora_dividir_outliers.csv

Division completada sin errores.


In [12]:
def stats(d):
    def ordinal(x):
        p = x.str.extract(r"^Q([1-4])\s+(\d{4})$")
        return (pd.to_numeric(p[1], errors="coerce") * 4
                + pd.to_numeric(p[0], errors="coerce"))
    dur = ordinal(d["end_date"]) - ordinal(d["start_date"]) + 1
    return pd.Series({
        "filas (experiencias)": len(d),
        "personas": d["resume_id"].nunique(),
        "mediana dur_Q": dur.median(),
        "máximo dur_Q": dur.max(),
        "vigentes ('Present')": int(d["end_date"].eq("Present").sum()),
    })

con = pd.read_csv(PROC / "empleos_limpio.csv", dtype=str, encoding="utf-8",
                  keep_default_na=False, na_values=[""])
sin = pd.read_csv(PROC / "empleos_limpio_sin_outliers.csv", dtype=str, encoding="utf-8",
                  keep_default_na=False, na_values=[""])
display(pd.DataFrame({"con cola larga": stats(con), "sin cola larga": stats(sin)}))

,con cola larga,sin cola larga
filas (experiencias),376567.0,341090.0
personas,71061.0,69182.0
mediana dur_Q,5.0,5.0
máximo dur_Q,160.0,24.0
vigentes ('Present'),18956.0,18956.0


## §8 Cierre y siguiente fase

**Resultados persistidos por esta unidad (verificados por asserts):**

- `data/03_processed/empleos_limpio.csv` — 376.567 filas × 14 columnas (banderas + re-categorización);
- `data/03_processed/empleos_limpio_sin_outliers.csv` — 341.090 filas × 14 (sin cola larga);
- `logs/bitacora_limpieza_empleos.csv` y `logs/bitacora_dividir_outliers.csv` — trazabilidad.

**Siguiente fase (transformación + minería):** derivación de secuencias por persona (estados: ocupación ISCO, `unknown`, vigencia), cálculo de transiciones y minería de patrones de trayectoria — a partir de estas dos versiones, para evaluar impactos.

Referencias: `README.md`, `METODOLOGIA_KDD_CRISPDM.md` (solo en disco), y el contexto de las sesiones 1–3 (fuera de versionado por decisión acordada).